In [1]:
import os
from pathlib import Path
import pickle
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import earthaccess

import numpy as np
import pandas as pd
import xarray as xr
import earthaccess
from tqdm.notebook import tqdm

In [2]:
PROJ_DIR = Path.cwd().parent.parent
DATA_DIR = PROJ_DIR / 'data'
lookup_fp = DATA_DIR / "lookup_station_times.pkl"
station_fp = DATA_DIR / 'station_coords.pkl'

In [3]:
# Load station -> [timestamps]
with open(lookup_fp, "rb") as f:
    lookup_station_times = pickle.load(f)

# Load station -> {lat, lon}
with open(station_fp, "rb") as f:
    station_coords = pickle.load(f)

# Quick diagnostics
stations_times = set(lookup_station_times.keys())
stations_coords = set(station_coords.keys())

missing_coords = sorted(stations_times - stations_coords)
missing_times  = sorted(stations_coords - stations_times)

print(f"Stations with times but no coords: {len(missing_coords)}")
if missing_coords[:10]:
    print("  examples:", missing_coords[:10])

print(f"Stations with coords but no times: {len(missing_times)}")
if missing_times[:10]:
    print("  examples:", missing_times[:10])

# Build the merged matchup driver table
rows = []
for station, times in lookup_station_times.items():
    if station not in station_coords:
        continue  # skip or handle separately

    lat = station_coords[station]["lat"]
    lon = station_coords[station]["lon"]

    for t in times:
        # Normalize to UTC pandas Timestamp
        t = pd.to_datetime(t, utc=True)
        rows.append({"station": station, "time": t, "lat": lat, "lon": lon})

stations_df = (
    pd.DataFrame(rows)
      .drop_duplicates(subset=["station", "time"])
      .sort_values(["station", "time"])
      .reset_index(drop=True)
)

stations_df.head(), stations_df.shape

Stations with times but no coords: 0
Stations with coords but no times: 0


(  station                      time      lat       lon
 0     BBB 2024-03-18 19:50:00+00:00  38.3126 -123.0825
 1     BBB 2024-05-16 18:30:00+00:00  38.3126 -123.0825
 2     BBB 2024-08-28 21:19:00+00:00  38.3126 -123.0825
 3     BBB 2024-09-30 16:11:00+00:00  38.3126 -123.0825
 4     BBB 2025-01-17 18:37:00+00:00  38.3126 -123.0825,
 (670, 4))

In [4]:
# 1. Login to Earthdata
auth = earthaccess.login()

In [5]:
def log_pace_granules_resumable_subset(
    df_subset,
    log_filename,
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1,
):
    """
    Resumable, append-only logger for a provided dataframe subset.
    Requirements:
      - df_subset has columns: lat, lon, time
      - df_subset has a stable integer column: row_id (0..n-1)
    Assumes earthaccess.login() already done by caller.
    """

    if "row_id" not in df_subset.columns:
        raise ValueError("df_subset must include a stable 'row_id' column (e.g., after reset_index).")

    done_rows = set()
    if os.path.exists(log_filename):
        prev = pd.read_csv(log_filename, usecols=["row_id", "status"])
        done_rows = set(prev.loc[prev["status"] == "ROW_DONE", "row_id"].unique())
        print(f"[resume] {os.path.basename(log_filename)}: {len(done_rows)} rows complete")

    write_header = not os.path.exists(log_filename)

    buffer = []
    rows_since_flush = 0

    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=os.path.basename(log_filename)):
        rid = int(row["row_id"])
        if rid in done_rows:
            continue

        lat0 = float(row["lat"])
        lon0 = float(row["lon"])
        t_insitu = pd.to_datetime(row["time"], utc=True)

        t0 = t_insitu - pd.Timedelta(hours=time_window_hours)
        t1 = t_insitu + pd.Timedelta(hours=time_window_hours)

        results = earthaccess.search_data(
            short_name="PACE_OCI_L2_AOP",
            temporal=(t0.to_pydatetime(), t1.to_pydatetime()),
            bounding_box=(lon0 - bbox_deg, lat0 - bbox_deg, lon0 + bbox_deg, lat0 + bbox_deg),
            cloud_hosted=True,
        )

        if len(results) == 0:
            buffer.append({
                "row_id": rid,
                "time": t_insitu,
                "lat": lat0,
                "lon": lon0,
                "granule_id": "NONE",
                "status": "No Granules Found",
            })
        else:
            for g in results:
                g_id = (
                    getattr(g, "producer_granule_id", None)
                    or getattr(g, "title", None)
                    or str(g)
                )

                dt = None
                remote = None
                status = None

                try:
                    remote = earthaccess.open([g], provider="OB_CLOUD")[0]
                    dt = xr.open_datatree(remote)

                    lat2d = dt["navigation_data"]["latitude"]
                    lon2d = dt["navigation_data"]["longitude"]

                    lat_min = float(lat2d.min().values)
                    lat_max = float(lat2d.max().values)
                    lon_min = float(lon2d.min().values)
                    lon_max = float(lon2d.max().values)

                    status = (
                        "Swath Covers Station"
                        if (lat_min <= lat0 <= lat_max and lon_min <= lon0 <= lon_max)
                        else "Swath Misses Station"
                    )

                except Exception as e:
                    status = f"Error: {type(e).__name__}: {e}"

                finally:
                    try:
                        if dt is not None:
                            dt.close()
                    except Exception:
                        pass
                    dt = None
                    remote = None

                buffer.append({
                    "row_id": rid,
                    "time": t_insitu,
                    "lat": lat0,
                    "lon": lon0,
                    "granule_id": g_id,
                    "status": status,
                })

        buffer.append({
            "row_id": rid,
            "time": t_insitu,
            "lat": lat0,
            "lon": lon0,
            "granule_id": "",
            "status": "ROW_DONE",
        })

        rows_since_flush += 1

        if rows_since_flush >= flush_every_rows:
            pd.DataFrame(buffer).to_csv(log_filename, mode="a", header=write_header, index=False)
            write_header = False
            buffer.clear()
            rows_since_flush = 0

    if buffer:
        pd.DataFrame(buffer).to_csv(log_filename, mode="a", header=write_header, index=False)

    return log_filename


In [6]:
stations_df.station.value_counts()

station
BML    104
CPP     98
SIO     93
SW      92
NP      74
MBB     73
MBF     73
TBB     18
T00     18
T16     17
BBB     10
Name: count, dtype: int64

In [ ]:
station = 'SIO'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
logfile = "SIO_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id


(49, 50)

In [8]:
station = "SIO"
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

df_sta.loc[df_sta.row_id == next_row_id, ["row_id", "time", "lat", "lon"]]


,row_id,time,lat,lon
50,50,2025-02-24 20:39:00+00:00,32.867,-117.257


In [11]:
# Pull the good granule names already coverd:
covers = log.loc[log.status == "Swath Covers Station", "granule_id"]
covers = covers[covers.notnull() & (covers != "NONE") & (covers != "")]
granules_useful = sorted(set(covers))

len(granules_useful), granules_useful[:10]


(55,
 ["Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 36.3242, 'Longitude': -107.57361}, {'Latitude': 30.86555, 'Longitude': -135.88306}, {'Latitude': 8.13026, 'Longitude': -128.43558}, {'Latitude': 13.13912, 'Longitude': -104.42388}, {'Latitude': 36.3242, 'Longitude': -107.57361}]}}]}}}\nTemporal coverage: {'RangeDateTime': {'EndingDateTime': '2024-04-15T20:45:17Z', 'BeginningDateTime': '2024-04-15T20:40:17Z'}}\nSize(MB): 158.6797866821289\nData: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240415T204017.L2.OC_AOP.V3_1.nc']",
  "Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 38.6699, 'Longitude': -109.1235}, {'Latitude': 33.13206, 'Longitude': -138.19722}, {'Latitude': 15.66324, 'Longitude': -131.

In [12]:
df_sta

,station,time,lat,lon,row_id
0,SIO,2024-03-11 19:15:00+00:00,32.867,-117.257,0
1,SIO,2024-03-18 19:00:00+00:00,32.867,-117.257,1
2,SIO,2024-03-25 19:34:00+00:00,32.867,-117.257,2
3,SIO,2024-04-01 19:27:00+00:00,32.867,-117.257,3
4,SIO,2024-04-08 19:53:00+00:00,32.867,-117.257,4
...,...,...,...,...,...
88,SIO,2025-11-17 20:01:00+00:00,32.867,-117.257,88
89,SIO,2025-12-01 20:49:00+00:00,32.867,-117.257,89
90,SIO,2025-12-08 20:17:00+00:00,32.867,-117.257,90
91,SIO,2025-12-15 20:06:00+00:00,32.867,-117.257,91


In [ ]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename="SIO_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile


[resume] SIO_pace_diagnostics.csv: 50 rows complete


SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# 3rd pass
station = "SIO"
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index


logfile = f"{station}_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id

(83, 84)

In [9]:
df_sta.loc[df_sta.row_id == next_row_id, ["row_id", "time", "lat", "lon"]]

,row_id,time,lat,lon
84,84,2025-10-20 21:21:00+00:00,32.867,-117.257


In [10]:
# Pull the good granule names already coverd:
covers = log.loc[log.status == "Swath Covers Station", "granule_id"]
covers = covers[covers.notnull() & (covers != "NONE") & (covers != "")]
granules_useful = sorted(set(covers))

len(granules_useful), granules_useful[:10]

(100,
 ["Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 34.60976, 'Longitude': -106.07507}, {'Latitude': 29.17929, 'Longitude': -133.90425}, {'Latitude': 11.6349, 'Longitude': -128.06076}, {'Latitude': 16.75861, 'Longitude': -103.69022}, {'Latitude': 34.60976, 'Longitude': -106.07507}]}}]}}}\nTemporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-02-24T20:34:56Z', 'EndingDateTime': '2025-02-24T20:39:55Z'}}\nSize(MB): 445.04975414276123\nData: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250224T203456.L2.OC_AOP.V3_1.nc']",
  "Collection: {'ShortName': 'PACE_OCI_L2_AOP', 'Version': '3.1'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 36.30452, 'Longitude': -100.69791}, {'Latitude': 30.84549, 'Longitude': -129.04016}, {'Latitude': 13.32932, 'Longitude':

In [11]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile

[resume] SIO_pace_diagnostics.csv: 84 rows complete


SIO_pace_diagnostics.csv:   0%|          | 0/93 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

'SIO_pace_diagnostics.csv'

In [12]:
log = pd.read_csv("SIO_pace_diagnostics.csv", usecols=["row_id","status"])
n_rows = log.loc[log.status == "ROW_DONE", "row_id"].nunique()
n_rows

93

BBB

In [13]:
station = 'BBB'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

BBB_pace_diagnostics.csv:   0%|          | 0/10 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
stations_df.station.value_counts()

station
BML    104
CPP     98
SIO     93
SW      92
NP      74
MBB     73
MBF     73
TBB     18
T00     18
T16     17
BBB     10
Name: count, dtype: int64

In [ ]:
station = 'BML'
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index

logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

BML_pace_diagnostics.csv:   0%|          | 0/104 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
station = 'BML'
log = pd.read_csv(f"{station}_pace_diagnostics.csv", usecols=["row_id","status"])
n_rows = log.loc[log.status == "ROW_DONE", "row_id"].nunique()
n_rows

86

In [8]:
df_sta = stations_df[stations_df.station == station].copy()
df_sta = df_sta.reset_index(drop=True)
df_sta["row_id"] = df_sta.index


logfile = f"{station}_pace_diagnostics.csv"
log = pd.read_csv(logfile)

done_rows = log.loc[log.status == "ROW_DONE", "row_id"]
last_done = int(done_rows.max()) if len(done_rows) else -1
next_row_id = last_done + 1

last_done, next_row_id

(85, 86)

In [9]:
logfile = log_pace_granules_resumable_subset(
    df_sta,
    log_filename=f"{station}_pace_diagnostics.csv",
    time_window_hours=3,
    bbox_deg=1.0,
    flush_every_rows=1
)

logfile

[resume] BML_pace_diagnostics.csv: 86 rows complete


BML_pace_diagnostics.csv:   0%|          | 0/104 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

'BML_pace_diagnostics.csv'


---


### Build matchup table

In [13]:
import os
import psutil
import logging
from pathlib import Path
import re
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import xarray as xr
import earthaccess

In [2]:
earthaccess.login()

In [15]:
def mem_mb():
    try:
        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    except Exception:
        return None
        
def append_chunk_to_netcdf(ds_chunk, out_nc_path):
    """
    Append along obs. If file doesn't exist, create it.
    Requires ds_chunk has dims: obs, wavelength
    """
    if not os.path.exists(out_nc_path):
        ds_chunk.to_netcdf(out_nc_path, mode="w")
    else:
        ds_chunk.to_netcdf(out_nc_path, mode="a", append_dim="obs")

def load_done_obs(progress_csv):
    """
    Returns a set of obs_keys already completed.
    obs_key is string: f"{pace_filename}||{row_id}"
    """
    if not os.path.exists(progress_csv):
        return set()
    prev = pd.read_csv(progress_csv, usecols=["obs_key", "status"])
    return set(prev.loc[prev["status"] == "OBS_DONE", "obs_key"].dropna().unique())

def log_progress(rows, progress_csv):
    """
    rows: list[dict] appended to CSV
    """
    df = pd.DataFrame(rows)
    header = not os.path.exists(progress_csv)
    df.to_csv(progress_csv, mode="a", header=header, index=False)

def get_granule_groups(
    diagnostics_csv: str,
    *,
    status_ok: str = "Swath Covers Station",
    short_name: str = "PACE_OCI_L2_AOP",
    version_tag: str = "V3_1",
    s3_bucket: str = "ob-cumulus-prod-public",
    require_columns=("row_id", "time", "lat", "lon", "granule_id"),
    keep_columns=("row_id", "time", "lat", "lon", "granule_id", "station"),
):
    """
    Build (pace_filename, gdf) groups from a diagnostics CSV.

    Returns:
      granule_groups: list of (pace_filename, gdf)
      m: cleaned matchup table (rows with status == status_ok)
    """

    diag = pd.read_csv(diagnostics_csv)

    # Validate required columns
    missing_cols = [c for c in require_columns if c not in diag.columns]
    if missing_cols:
        raise ValueError(f"Diagnostics file missing required columns: {missing_cols}")

    # Filter to valid coverage rows
    m = diag[diag["status"] == status_ok].copy()

    # Keep only columns that exist
    cols = [c for c in keep_columns if c in m.columns]
    m = m[cols].copy()

    # Parse time (keep UTC)
    m["time"] = pd.to_datetime(m["time"], utc=True, errors="coerce")

    # Extract filename whether granule_id is filename OR str(DataGranule) blob
    # Example filename: PACE_OCI.20250117T194803.L2.OC_AOP.V3_1.nc
    pat = rf"(PACE_OCI\.\d{{8}}T\d{{6}}\.L2\.OC_AOP\.{re.escape(version_tag)}\.nc)"
    m["pace_filename"] = m["granule_id"].astype(str).str.extract(pat, expand=False)

    # Fail-fast if we can't extract names
    bad = m[m["pace_filename"].isna()]
    if len(bad) > 0:
        sample = bad[["row_id", "granule_id"]].head(3).to_dict("records")
        raise ValueError(
            f"Could not extract pace_filename for {len(bad)} rows. "
            f"Sample: {sample}"
        )

    # Build S3 URI (we open directly; no extra CMR search needed)
    m["s3_uri"] = "s3://" + s3_bucket + "/" + m["pace_filename"]

    # Group by swath file
    granule_groups = list(m.groupby("pace_filename", sort=True))

    return granule_groups, m



def get_file_logger(log_path: str, name: str = "rrs_extract"):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False  # don't duplicate in notebook

    # avoid double handlers if you re-run cells
    if not any(isinstance(h, logging.FileHandler) and h.baseFilename == str(Path(log_path).resolve())
               for h in logger.handlers):
        fh = logging.FileHandler(log_path)
        fh.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger


Loop granules → open once → for each row do nearest pixel → 3×3 mean/std → append to NetCDF

In [4]:
def run_rrs_extraction_resumable(
    granule_groups,
    out_nc,
    progress_csv,
    *,
    buffer_n=2,                 # <-- set to 1 or 2 for frequent writing
    window=5,                   # <-- single 5x5 window (simpler)
    provider="OB_CLOUD",
    log_path="extract.log",
    clear_file_cache_each_granule=True,
):
    """
    Resumable extraction with file logging and cache control.
    Assumes earthaccess.login() already called.
    """

    logger = get_file_logger(log_path)
    xr.set_options(file_cache_maxsize=0)

    done_obs = load_done_obs(progress_csv)
    logger.info(f"START | out_nc={out_nc} progress={progress_csv} done_obs={len(done_obs)} buffer_n={buffer_n} window={window}")
    mm = mem_mb()
    if mm is not None:
        logger.info(f"MEM_MB start: {mm:.1f}")

    wavelength_ref = None
    half = window // 2

    # buffers
    buf_times, buf_lats, buf_lons = [], [], []
    buf_rowid, buf_gran, buf_obskey = [], [], []
    buf_rrs_mean, buf_rrs_std = [], []
    progress_buffer = []

    def flush():
        if len(buf_times) == 0:
            return

        ds_chunk = xr.Dataset(
            data_vars={
                "Rrs_mean": (("obs", "wavelength"), np.stack(buf_rrs_mean, axis=0)),
                "Rrs_std":  (("obs", "wavelength"), np.stack(buf_rrs_std,  axis=0)),
            },
            coords={
                "time": ("obs", buf_times),
                "lat": ("obs", np.array(buf_lats, dtype=np.float32)),
                "lon": ("obs", np.array(buf_lons, dtype=np.float32)),
                "row_id": ("obs", np.array(buf_rowid, dtype=np.int64)),
                "granule_id": ("obs", np.array(buf_gran, dtype=object)),
                "obs_key": ("obs", np.array(buf_obskey, dtype=object)),
                "wavelength": ("wavelength", wavelength_ref),
            },
            attrs={"rrs_aggregation": f"{window}x{window}_mean_std"},
        )

        append_chunk_to_netcdf(ds_chunk, out_nc)
        log_progress(progress_buffer, progress_csv)

        logger.info(f"FLUSH | wrote_obs={len(buf_times)} total_done_obs={len(done_obs)}")
        mm2 = mem_mb()
        if mm2 is not None:
            logger.info(f"MEM_MB after flush: {mm2:.1f}")

        buf_times.clear(); buf_lats.clear(); buf_lons.clear()
        buf_rowid.clear(); buf_gran.clear(); buf_obskey.clear()
        buf_rrs_mean.clear(); buf_rrs_std.clear()
        progress_buffer.clear()

    # loop granules
    for pace_filename, gdf in tqdm(granule_groups, desc="granules"):
        s3_uri = gdf["s3_uri"].iloc[0]
        logger.info(f"GRANULE start | {pace_filename} | rows={len(gdf)} | mem_mb={mem_mb()}")

        remote = None
        dt = None

        # counters per granule
        n_skip_done = n_no_pix = n_no_spec = n_written = 0

        try:
            remote = earthaccess.open([s3_uri], provider=provider)[0]
            dt = xr.open_datatree(remote)

            lat2d = dt["navigation_data"]["latitude"]
            lon2d = dt["navigation_data"]["longitude"]
            Rrs   = dt["geophysical_data"]["Rrs"]
            wv    = dt["sensor_band_parameters"]["wavelength_3d"].values.astype(np.float32)

            if wavelength_ref is None:
                wavelength_ref = wv
                logger.info(f"WAVELENGTH set | n={len(wavelength_ref)}")
            else:
                if len(wv) != len(wavelength_ref) or np.nanmax(np.abs(wv - wavelength_ref)) > 1e-6:
                    logger.info(f"GRANULE skip wavelength mismatch | {pace_filename}")
                    log_progress([{"obs_key": f"{pace_filename}||ALL", "status": "SKIP_WAVELENGTH_MISMATCH"}], progress_csv)
                    continue

            for _, row in gdf.iterrows():
                row_id = int(row["row_id"])
                obs_key = f"{pace_filename}||{row_id}"
                if obs_key in done_obs:
                    n_skip_done += 1
                    continue

                lat0 = float(row["lat"])
                lon0 = float(row["lon"])
                t0   = pd.to_datetime(row["time"], utc=True)

                # nearest pixel (creates a big temporary when .values is called)
                dist2 = (lat2d - lat0)**2 + (lon2d - lon0)**2
                dist2 = dist2.where(lat2d.notnull() & lon2d.notnull())
                iy, ix = np.unravel_index(np.nanargmin(dist2.values), dist2.shape)

                patch = Rrs[:, iy-half:iy+half+1, ix-half:ix+half+1]
                if not np.isfinite(patch.values).any():
                    progress_buffer.append({"obs_key": obs_key, "status": f"NO_VALID_PIXELS_{window}x{window}"})
                    n_no_pix += 1
                    continue

                patch = patch.where(patch.notnull())
                rrs_mean = patch.mean(dim=("number_of_lines", "pixels_per_line"), skipna=True).values
                rrs_std  = patch.std(dim=("number_of_lines", "pixels_per_line"), skipna=True).values

                if not np.isfinite(rrs_mean).any():
                    progress_buffer.append({"obs_key": obs_key, "status": "NO_VALID_SPECTRUM"})
                    n_no_spec += 1
                    continue

                # buffer
                buf_times.append(t0.to_datetime64())
                buf_lats.append(lat0); buf_lons.append(lon0)
                buf_rowid.append(row_id)
                buf_gran.append(pace_filename)
                buf_obskey.append(obs_key)
                buf_rrs_mean.append(rrs_mean.astype(np.float32))
                buf_rrs_std.append(rrs_std.astype(np.float32))

                progress_buffer.append({"obs_key": obs_key, "status": "OBS_DONE"})
                done_obs.add(obs_key)
                n_written += 1

                if len(buf_times) >= buffer_n:
                    flush()

            logger.info(
                f"GRANULE done | {pace_filename} | written={n_written} "
                f"skip_done={n_skip_done} no_pix={n_no_pix} no_spec={n_no_spec}"
            )

        except Exception as e:
            logger.exception(f"GRANULE error | {pace_filename} | {type(e).__name__}: {e}")
            # flush whatever you have so far before continuing/crashing
            flush()
            raise

        finally:
            try:
                if dt is not None:
                    dt.close()
            except Exception:
                pass
            dt = None
            remote = None

            if clear_file_cache_each_granule:
                try:
                    xr.backends.file_manager.FILE_CACHE.clear()
                except Exception:
                    pass

    flush()
    logger.info("DONE")
    mm_end = mem_mb()
    if mm_end is not None:
        logger.info(f"MEM_MB end: {mm_end:.1f}")

In [6]:
xr.set_options(file_cache_maxsize=0)

SITE = 'BBB'

diag_path = f"{SITE}_pace_diagnostics.csv"  # <-- set this
granule_groups, m = get_granule_groups(diag_path)

len(granule_groups), m.shape, granule_groups[0][0]


(11, (11, 7), 'PACE_OCI.20240318T204706.L2.OC_AOP.V3_1.nc')

In [12]:
run_rrs_extraction_resumable(
    granule_groups=granule_groups,
    out_nc=f"{SITE}_rrs_matchups_3x3.nc",
    progress_csv=f"{SITE}_extract_progress.csv",
    buffer_n=2,  # write every 2 valid spectra
    window=5,
    log_path = f"{SITE}_extract.log"
)


granules:   0%|          | 0/11 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]